# 🎮 Visual Rollout of Best Trained PPO Model (Pygame)

This notebook is dedicated exclusively to discovering and running an **interactive Pygame graphical simulation** of the best trained PPO checkpoint.

### Instructions:
1. Run the cell below to automatically detect the top checkpoint saved in `experiments/ray_results/`.
2. An interactive Pygame window will open, rendering the real-time rollout of the agents in Overcooked MARL.

In [ ]:
import os
import json
import sys
import pygame
import ray
import numpy as np
from pathlib import Path
from ray.rllib.algorithms.algorithm import Algorithm
from ray.rllib.env.multi_agent_env import MultiAgentEnv
from ray.tune.registry import register_env

# Add project root to sys.path
project_root = Path("..").resolve().parent if Path("..").resolve().name == "experiments" else Path(".").resolve().parent
sys.path.insert(0, str(project_root))

try:
    from overcooked_marl_env import OvercookedMARL
except ImportError:
    from env import OvercookedMARL

class ParallelEnvToMultiAgent(MultiAgentEnv):
    def __init__(self, parallel_env):
        self.env = parallel_env
        self.agents = self.env.possible_agents
        super().__init__()
        
    @property
    def observation_space(self):
        return self.env.observation_space

    @property
    def action_space(self):
        return self.env.action_space

    def reset(self, *, seed=None, options=None):
        return self.env.reset(seed=seed, options=options)

    def step(self, action_dict):
        return self.env.step(action_dict)

    def render(self):
        return self.env.render()
    
    def close(self):
        return self.env.close()

ENVIRONMENT_CONFIG = {
    "map_path": "default_map.txt",
    "num_fixed_agents": 1,
    "featurized_obs": True
}

def env_creator_fn():
    return OvercookedMARL(
        render_mode=None,
        map_path=ENVIRONMENT_CONFIG["map_path"],
        num_fixed_agents=ENVIRONMENT_CONFIG["num_fixed_agents"],
        verbose=False,
        featurized_obs=ENVIRONMENT_CONFIG["featurized_obs"]
    )

def discover_best_checkpoint(ray_results_dir="../ray_results"):
    results_path = Path(ray_results_dir).resolve()
    if not results_path.exists():
        results_path = Path("experiments/ray_results").resolve()
        
    checkpoints = []
    for cp_path in results_path.glob("**/checkpoint_*"):
        if cp_path.is_dir() and (cp_path / "rllib_checkpoint.json").exists():
            try:
                model_info_path = cp_path.parents[1] / "model_info.json"
                avg_reward = 0.0
                if model_info_path.exists():
                    with open(model_info_path, "r") as f:
                        meta = json.load(f)
                        avg_reward = meta.get("avg_reward", 0.0)
                checkpoints.append((cp_path, avg_reward))
            except Exception:
                checkpoints.append((cp_path, 0.0))
                
    checkpoints.sort(key=lambda x: (x[1], int(x[0].name.split("_")[-1]) if x[0].name.split("_")[-1].isdigit() else 0), reverse=True)
    return checkpoints[0][0] if checkpoints else None

def run_pygame_visual_rollout(checkpoint_path, num_episodes=1, max_steps=800):
    print(f"[INFO] Loading checkpoint: {checkpoint_path}")
    register_env("overcooked_marl_parallel", lambda config: ParallelEnvToMultiAgent(env_creator_fn()))
    
    ray.init(ignore_reinit_error=True)
    algorithm = Algorithm.from_checkpoint(str(checkpoint_path))
    
    env = ParallelEnvToMultiAgent(OvercookedMARL(
        render_mode="human",
        map_path=ENVIRONMENT_CONFIG["map_path"],
        num_fixed_agents=ENVIRONMENT_CONFIG["num_fixed_agents"],
        verbose=False,
        featurized_obs=ENVIRONMENT_CONFIG["featurized_obs"]
    ))
    
    try:
        for episode in range(num_episodes):
            obs_dict, infos = env.reset(seed=42 + episode)
            episode_reward = 0.0
            
            for step in range(max_steps):
                actions = {}
                for agent_id, obs in obs_dict.items():
                    policy_id = f"policy_{agent_id}"
                    try:
                        action = algorithm.compute_single_action(obs, policy_id=policy_id, explore=False)
                    except Exception:
                        action = env.action_space[agent_id].sample()
                    actions[agent_id] = action
                    
                obs_dict, rewards, terminations, truncations, infos = env.step(actions)
                episode_reward += sum(rewards.values())
                env.render()
                pygame.time.wait(20)
                
                if any(terminations.values()) or any(truncations.values()):
                    break
            print(f"[SUCCESS] Episode {episode+1} finished. Total Reward: {episode_reward:+.2f}")
    finally:
        env.close()
        ray.shutdown()

best_cp = discover_best_checkpoint()
if best_cp:
    run_pygame_visual_rollout(best_cp, num_episodes=1)
else:
    print("[ERROR] No checkpoints found in ray_results/")
